# Symptom Dataset EDA
In-depth investigation of medical symptom mapping and disease classification feasibility.

## 1. Environment Setup
Loading essential libraries and setting visualization parameters.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
%matplotlib inline

# Aesthetics
sns.set(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 2. Data Loading and Initial Inspection

In [ ]:
RAW_PATH = '../../data/raw/symptoms/final_symptoms_to_disease.csv'
df = pd.read_csv(RAW_PATH)
print(f'Dataset Shape: {df.shape}')
df.head()

## 3. Disease Label Analysis
Analyzing class distribution and detecting potential imbalance.

### Pre-generated Distribution
![Disease Distribution](../../reports/figures/symptoms_disease_dist.png)

In [ ]:
disease_counts = df['diseases'].value_counts()
print(f'Total unique diseases: {len(disease_counts)}')

# Visualize Top 30 Diseases
disease_counts[:30].plot(kind='bar', color='skyblue')
plt.title('Top 30 Diseases by Frequency')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45, ha='right')
plt.show()

## 4. Symptom Extraction and Frequency

### Pre-generated Top Symptoms
![Top Symptoms](../../reports/figures/symptoms_top_symptoms.png)

In [ ]:
all_symptoms = []
df['symptom_list'] = df['symptom_text'].apply(lambda x: [s.strip().lower() for s in str(x).split(',')])
for s_list in df['symptom_list']:
    all_symptoms.extend(s_list)

symptom_counts = Counter(all_symptoms)
symptom_df = pd.DataFrame(symptom_counts.most_common(30), columns=['Symptom', 'Count'])

sns.barplot(data=symptom_df, x='Count', y='Symptom', palette='viridis')
plt.title('Top 30 Most Frequent Symptoms')
plt.show()

## 5. Advanced Analysis: Symptom Co-occurrence

### Pre-generated Co-occurrence Matrix
![Co-occurrence](../../reports/figures/symptoms_cooccurrence.png)

In [ ]:
top_20_symptoms = [s for s, c in symptom_counts.most_common(20)]
co_matrix = np.zeros((20, 20))
for s_list in df['symptom_list']:
    for i in range(20):
        for j in range(20):
            if top_20_symptoms[i] in s_list and top_20_symptoms[j] in s_list:
                co_matrix[i, j] += 1

sns.heatmap(co_matrix, xticklabels=top_20_symptoms, yticklabels=top_20_symptoms, annot=True, fmt='g', cmap='YlGnBu')
plt.title('Symptom Co-occurrence Heatmap')
plt.show()

## 6. Disease Overlap (Jaccard Similarity)

### Pre-generated Similarity Matrix
![Disease Similarity](../../reports/figures/symptoms_disease_similarity.png)

In [ ]:
disease_symptom_map = df.groupby('diseases')['symptom_list'].apply(lambda x: set().union(*x)).to_dict()
diseases = list(disease_symptom_map.keys())

# Analyze Top 20 for visibility
sub_diseases = diseases[:20]
overlap_matrix = np.zeros((20, 20))
for i in range(20):
    for j in range(20):
        s1 = disease_symptom_map[sub_diseases[i]]
        s2 = disease_symptom_map[sub_diseases[j]]
        if len(s1.union(s2)) > 0:
            overlap_matrix[i, j] = len(s1.intersection(s2)) / len(s1.union(s2))

sns.heatmap(overlap_matrix, xticklabels=sub_diseases, yticklabels=sub_diseases, annot=True, fmt='.2f', cmap='Reds')
plt.title('Disease Symptom Overlap (Jaccard Similarity)')
plt.show()

## 7. Conclusions and Model Implications
- **Imbalance**: Class weights or oversampling may be needed for rare diseases.
- **Ambiguity**: High overlap suggests Top-K prediction is safer than Top-1.
- **Sparsity**: Many symptoms are rare; TF-IDF helps focus on distinguishing features.